# Sprint 1: Infraestructura de Datos y Pipeline de Calibracion

> **Prerrequisitos:** Solo requiere ejecutar el **Bloque 0** (configuracion de entorno y parches).
> Este sprint es el punto de entrada del proyecto.

**Objetivo:** Consolidar el entorno de datos local en WSL2 y construir el cargador de tensores
que alimentara los algoritmos de cuantizacion offline (GPTQ y AWQ) en la RTX 4060 Ti.

**Dataset:** [fsicoli/common_voice_17_0](https://huggingface.co/datasets/fsicoli/common_voice_17_0) (config `es`)
- Mirror comunitario completo de Mozilla Common Voice 17.0 Spanish (~2,220h total, 562h validadas)
- Splits nativos: `train`, `validation`, `test`, `other`, `invalidated`
- Audio original a 16kHz mono, licencia CC0-1.0
- **Motivo del cambio:** Mozilla retiro CV 17.0 de HuggingFace en Oct 2025. El mirror `fsicoli` almacena los audios directamente.

> **Nota tecnica:** VibeVoice 1.5B procesa audio crudo a **24kHz mono**, no espectrogramas de Mel.
> El preprocesamiento consiste en remuestreo + normalizacion de amplitud, no en conversion frecuencial.

## Orden de ejecucion: Sprint 1 y Sprint 1.5

Ejecute los identificadores en este orden. No avance a Sprints 2-7 hasta que `S1.5-07` apruebe el gate WER.

| Orden | Entorno | Accion |
|---|---|---|
| `S1-00` | Notebook | Entorno, rutas y parche CONFIG_MAPPING |
| `S1-01` | Notebook | Descargar Common Voice 17 ES |
| `S1-02` | Notebook | Exploracion opcional del dataset |
| `S1-03` | Notebook | Cargar y asignar splits nativos |
| `S1-04` | Notebook | Guardar trazabilidad de splits |
| `S1-05` | Notebook | Definir y validar preprocesamiento |
| `S1-06` | Notebook | Construir calibracion estratificada |
| `S1-07` | Notebook | Guardar tensor y metadatos |
| `S1-08` | Notebook | Verificar calibracion |
| `S1.5-01` | Notebook | Crear train/dev speaker-disjoint |
| `S1.5-02` | Notebook | Validar script de entrenamiento |
| `S1.5-03` | Notebook | Ejecutar `scripts/run_finetune_es.sh` y validar el LoRA |
| `S1.5-04` | Notebook | Validar script de merge |
| `S1.5-05` | Notebook | Ejecutar `scripts/merge_es_checkpoint.sh` y validar el checkpoint |
| `S1.5-06` | Notebook | Cargar checkpoint BF16 corregido |
| `S1.5-07` | Notebook | Gate linguistico WER/CER y WAV |


---
## Bloque 0: Configuracion del Entorno y Parches Criticos

Se aplica el parche obligatorio de `transformers` para evitar colisiones de mapeo
con el fork comunitario de VibeVoice (ver `agents.md` seccion 3).

In [1]:
print("[S1-00] Configuracion del entorno y rutas")
import sys
import os
import json
import time
import random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import torch
import transformers
from datasets import load_dataset, Audio, Dataset, DatasetDict, concatenate_datasets
from tqdm.auto import tqdm

# --- Parche critico: colision de mapeo en transformers ---
if hasattr(transformers, "CONFIG_MAPPING"):
    transformers.CONFIG_MAPPING._extra_content.pop("vibevoice_acoustic_tokenizer", None)
    if hasattr(transformers.CONFIG_MAPPING, "_mapping"):
        transformers.CONFIG_MAPPING._mapping.pop("vibevoice_acoustic_tokenizer", None)
print("Parche de CONFIG_MAPPING aplicado.")

# --- Rutas del proyecto ---
PROJECT_ROOT = Path(os.path.abspath("")).parent if "notebooks" in os.path.abspath("") else Path(os.path.abspath(""))
if (Path.cwd() / "data").exists():
    PROJECT_ROOT = Path.cwd()
else:
    PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
WEIGHTS_DIR = PROJECT_ROOT / "weights"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DATA_DIR     : {DATA_DIR}")
print(f"WEIGHTS_DIR  : {WEIGHTS_DIR}")
print(f"Torch        : {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total   : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

[S1-00] Configuracion del entorno y rutas


/home/alfrog/micromamba/envs/vibevoice/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Parche de CONFIG_MAPPING aplicado.
PROJECT_ROOT : /home/alfrog/projects/VibeVoice_Optimization
DATA_DIR     : /home/alfrog/projects/VibeVoice_Optimization/data
WEIGHTS_DIR  : /home/alfrog/projects/VibeVoice_Optimization/weights
Torch        : 2.12.0+cu130  |  CUDA: True
GPU          : NVIDIA GeForce RTX 4060 Ti
VRAM Total   : 8.00 GB


---
## TSK-1.1: Descarga Automatizada del Dataset

Se descarga el subset en espanol de Common Voice 17.0 desde el mirror comunitario
`fsicoli/common_voice_17_0` con config `"es"`. El dataset contiene las particiones
nativas `train`/`validation`/`test`/`other` con audios y transcripciones.

**Estrategia:** Cargamos el split `train` como base para el 70% de datos de
entrenamiento (fine-tuning), y los splits `test` y `validation` nativos como
nuestros sets de benchmark y calibracion respectivamente.

In [2]:
print("[S1-01] Descarga de Common Voice 17 espanol")
DATASET_ID = "fsicoli/common_voice_17_0"
DATASET_CONFIG = "es"

print(f"Descargando dataset: {DATASET_ID} (config={DATASET_CONFIG})")
print("(La primera ejecucion descargara los audios de espanol; puede tomar ~30-60 min)\n")

t_start = time.time()

cv17_dataset = load_dataset(
    DATASET_ID,
    DATASET_CONFIG,
    split="train",
    trust_remote_code=True,
)

t_elapsed = time.time() - t_start
print(f"\nDescarga completada en {t_elapsed/60:.1f} minutos.")
print(f"Registros cargados (train): {len(cv17_dataset):,}")
print(f"Columnas: {cv17_dataset.column_names}")

[S1-01] Descarga de Common Voice 17 espanol
Descargando dataset: fsicoli/common_voice_17_0 (config=es)
(La primera ejecucion descargara los audios de espanol; puede tomar ~30-60 min)


Descarga completada en 0.0 minutos.
Registros cargados (train): 336,846
Columnas: ['client_id', 'path', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment', 'variant']


---
## Exploracion del Dataset

Inspeccionamos la estructura, metadatos demograficos y distribucion de duraciones
para validar la calidad de la muestra antes de la particion.

In [3]:
print("[S1-02] Exploracion opcional del dataset")
# --- Muestra de registros ---
print("=== Ejemplos del dataset ===\n")
for i in range(3):
    sample = cv17_dataset[i]
    audio_info = sample["audio"]
    duration_s = len(audio_info["array"]) / audio_info["sampling_rate"]
    print(f"[{i}] \"{sample['sentence'][:80]}...\"")
    print(f"    Duracion: {duration_s:.2f}s | SR: {audio_info['sampling_rate']}Hz")
    print(f"    Locale: {sample['locale']} | Genero: {sample['gender']} | Edad: {sample['age']}")
    print()

# --- Distribucion de duraciones (muestreo estadistico) ---
SAMPLE_SIZE = 5000
indices = random.sample(range(len(cv17_dataset)), min(SAMPLE_SIZE, len(cv17_dataset)))
durations = []
for idx in tqdm(indices, desc="Muestreando duraciones"):
    arr = cv17_dataset[idx]["audio"]["array"]
    sr = cv17_dataset[idx]["audio"]["sampling_rate"]
    durations.append(len(arr) / sr)

durations = np.array(durations)
print(f"\n=== Estadisticas de Duracion (muestra n={len(durations)}) ===")
print(f"Media    : {durations.mean():.2f}s")
print(f"Mediana  : {np.median(durations):.2f}s")
print(f"Min/Max  : {durations.min():.2f}s / {durations.max():.2f}s")
print(f"Std      : {durations.std():.2f}s")
print(f"Horas est. totales: {durations.mean() * len(cv17_dataset) / 3600:.1f}h")

# --- Distribucion de genero y edad ---
genders = Counter()
ages = Counter()
for idx in tqdm(indices, desc="Muestreando demografia"):
    sample = cv17_dataset[idx]
    genders[sample["gender"]] += 1
    ages[sample["age"]] += 1

print(f"\nGeneros  : {dict(genders.most_common())}")
print(f"Edades    : {dict(ages.most_common())}")

[S1-02] Exploracion opcional del dataset
=== Ejemplos del dataset ===

[0] "A ese grupo de amigos les gusta comer helados por la tarde...."
    Duracion: 3.89s | SR: 48000Hz
    Locale: es | Genero: male_masculine | Edad: twenties

[1] "A mi profesora le importa demasiado la gramática y muy poco el contenido...."
    Duracion: 5.16s | SR: 48000Hz
    Locale: es | Genero: male_masculine | Edad: twenties

[2] "A decir verdad , eres demasiado genial ...."
    Duracion: 3.05s | SR: 48000Hz
    Locale: es | Genero: male_masculine | Edad: twenties



Muestreando duraciones: 100%|██████████| 5000/5000 [00:41<00:00, 121.21it/s]



=== Estadisticas de Duracion (muestra n=5000) ===
Media    : 5.17s
Mediana  : 5.08s
Min/Max  : 1.55s / 10.85s
Std      : 1.44s
Horas est. totales: 483.6h


Muestreando demografia: 100%|██████████| 5000/5000 [00:17<00:00, 288.27it/s]


Generos  : {'male_masculine': 2843, 'female_feminine': 1557, '': 600}
Edades    : {'twenties': 2327, 'thirties': 621, '': 583, 'sixties': 465, 'fifties': 422, 'teens': 292, 'fourties': 284, 'seventies': 5, 'eighties': 1}


---
## TSK-1.2: Particion de la Muestra

Common Voice 17.0 ya proporciona divisiones nativas. Las usamos directamente
para mantener trazabilidad con el estandar de la comunidad:

| Particion | Split CV17 | Proposito en el proyecto |
|-----------|-----------|---------------------------|
| Test      | `test`    | Benchmark final (Sprint 5) |
| Validacion| `validation` + muestra `other` | Calibracion offline (GPTQ/AWQ) |
| Train     | `train`   | Fine-tuning espanol (Sprint 1.5) |

Cargamos todos los splits y guardamos los indices para trazabilidad.

In [4]:
print("[S1-03] Carga y asignacion de splits nativos")
# --- Cargar todos los splits nativos de CV17 es ---
print("Cargando splits nativos de Common Voice 17.0 (es)...")

cv17_all = load_dataset(
    DATASET_ID,
    DATASET_CONFIG,
    trust_remote_code=True,
)

for split_name, split_data in cv17_all.items():
    print(f"  {split_name:<20} {len(split_data):>10,} registros")

train_dataset = cv17_all["train"]
test_dataset = cv17_all["test"]
val_dataset = cv17_all["validation"]
other_dataset = cv17_all.get("other", None)

print(f"\n=== Asignacion para el proyecto ===")
print(f"Train (fine-tuning)       : {len(train_dataset):,} registros")
print(f"Test (benchmark)          : {len(test_dataset):,} registros")
print(f"Validation (calibracion)  : {len(val_dataset):,} registros")
if other_dataset:
    print(f"Other (calibracion extra) : {len(other_dataset):,} registros")

[S1-03] Carga y asignacion de splits nativos
Cargando splits nativos de Common Voice 17.0 (es)...
  train                   336,846 registros
  validation               15,857 registros
  test                     15,857 registros
  other                 1,138,631 registros
  invalidated              86,227 registros

=== Asignacion para el proyecto ===
Train (fine-tuning)       : 336,846 registros
Test (benchmark)          : 15,857 registros
Validation (calibracion)  : 15,857 registros
Other (calibracion extra) : 1,138,631 registros


In [5]:
print("[S1-04] Guardado de trazabilidad de splits")
# --- Guardar metadatos de particiones para trazabilidad ---
partitions_meta = {
    "dataset_id": DATASET_ID,
    "dataset_config": DATASET_CONFIG,
    "train_count": len(train_dataset),
    "test_count": len(test_dataset),
    "val_count": len(val_dataset),
    "other_count": len(other_dataset) if other_dataset else 0,
    "splits_used": {
        "fine_tuning": "train",
        "benchmark_test": "test",
        "calibration": "validation (+ other)",
    },
}

partitions_path = DATA_DIR / "dataset_partitions.json"
with open(partitions_path, "w") as f:
    json.dump(partitions_meta, f, indent=2)
print(f"Metadatos de particiones guardados en: {partitions_path}")

[S1-04] Guardado de trazabilidad de splits
Metadatos de particiones guardados en: /home/alfrog/projects/VibeVoice_Optimization/data/dataset_partitions.json


---
## TSK-1.3: Preprocesamiento Acustico

VibeVoice 1.5B espera audio crudo monofonico a **24,000 Hz** (no espectrogramas de Mel).
El pipeline de preprocesamiento consiste en:

1. **Remuestreo:** 48,000 Hz → 24,000 Hz (usando `librosa.resample`)
2. **Conversion a mono:** garantizado por el dataset
3. **Normalizacion de amplitud:** a -25 dB FS (mismo criterio que `AudioNormalizer` del modelo)
4. **Validacion:** descarte de audios corruptos o de duracion minima (<0.5s)

El resultado se almacena como tensores `torch.float32` en archivos `.pt`.

In [6]:
print("[S1-05] Preprocesamiento de audio a 24 kHz y -25 dBFS")
import librosa

TARGET_SR = 24000
TARGET_DB_FS = -25.0
MIN_DURATION_S = 0.5
EPS = 1e-6

def preprocess_audio(audio_array: np.ndarray, orig_sr: int) -> np.ndarray | None:
    """
    Preprocesa un array de audio para VibeVoice:
    remuestreo a 24kHz + normalizacion a target_dB_FS.
    Retorna None si el audio es invalido.
    """
    if audio_array is None or len(audio_array) == 0:
        return None

    # Remuestreo a 24kHz
    if orig_sr != TARGET_SR:
        audio = librosa.resample(
            y=audio_array.astype(np.float32),
            orig_sr=orig_sr,
            target_sr=TARGET_SR,
        )
    else:
        audio = audio_array.astype(np.float32)

    duration = len(audio) / TARGET_SR
    if duration < MIN_DURATION_S:
        return None

    # Normalizacion a target dB FS
    rms = np.sqrt(np.mean(audio ** 2))
    if rms < EPS:
        return None
    scalar = (10 ** (TARGET_DB_FS / 20)) / (rms + EPS)
    audio = audio * scalar

    # Prevenir clipping
    peak = np.max(np.abs(audio))
    if peak > 1.0:
        audio = audio / peak

    return audio.astype(np.float32)

# --- Prueba con los primeros 5 registros ---
print("Validando preprocesamiento con 5 muestras...\n")
for i in range(5):
    sample = val_dataset[i]
    arr = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]
    processed = preprocess_audio(arr, sr)
    if processed is not None:
        print(f"[{i}] OK: {len(processed)/TARGET_SR:.2f}s -> {processed.shape}, "
              f"peak={np.max(np.abs(processed)):.3f}, rms={np.sqrt(np.mean(processed**2)):.4f}")
    else:
        print(f"[{i}] DESCARTADO")

[S1-05] Preprocesamiento de audio a 24 kHz y -25 dBFS
Validando preprocesamiento con 5 muestras...

[0] OK: 4.85s -> (116352,), peak=0.409, rms=0.0562
[1] OK: 5.57s -> (133632,), peak=0.453, rms=0.0562
[2] OK: 4.37s -> (104832,), peak=0.368, rms=0.0562
[3] OK: 5.18s -> (124416,), peak=0.358, rms=0.0562
[4] OK: 3.34s -> (80064,), peak=0.557, rms=0.0562


---
## TSK-1.4: Construccion del Set de Calibracion Offline

Se construye el tensor de calibracion a partir de la particion de validacion (10%).

**Estrategia:**
- Para GPTQ/AWQ se necesita un subconjunto representativo (tipicamente 128-512 muestras)
  que cubra diversidad de duraciones, generos y acentos.
- Se seleccionan `N_CALIB_SAMPLES` muestras con muestreo estratificado simple.
- Los audios preprocesados se concatenan en un unico tensor `[total_samples]`.
- Se guarda metadato con offsets para reconstruir limites de cada alocucion.

**Formato de salida:**
- `data/calibration_tensor.pt` — tensor float32 con audio concatenado
- `data/calibration_metadata.json` — indices originales, offsets, duraciones

In [7]:
print("[S1-06] Construccion de calibracion estratificada")
N_CALIB_SAMPLES = 512
CALIB_SEED = 123

random.seed(CALIB_SEED)
np.random.seed(CALIB_SEED)

print(f"Seleccionando {N_CALIB_SAMPLES} muestras de calibracion...")

# --- Muestreo estratificado por acento y genero, con candidatos de reserva ---
calib_source = val_dataset
n_source = len(calib_source)
meta_columns = [name for name in ["accent", "gender"] if name in calib_source.column_names]
meta_view = calib_source.select_columns(meta_columns)
calib_groups = defaultdict(list)
for idx in range(n_source):
    row = meta_view[idx]
    key = tuple((row.get(name) or "unknown") for name in meta_columns)
    calib_groups[key].append(idx)
rng_calib = random.Random(CALIB_SEED)
for indices_group in calib_groups.values():
    rng_calib.shuffle(indices_group)
candidate_target = min(n_source, max(N_CALIB_SAMPLES, int(N_CALIB_SAMPLES * 1.25)))
calib_candidate_indices = []
active_groups = list(calib_groups)
rng_calib.shuffle(active_groups)
while active_groups and len(calib_candidate_indices) < candidate_target:
    next_groups = []
    for key in active_groups:
        if calib_groups[key]:
            calib_candidate_indices.append(calib_groups[key].pop())
        if calib_groups[key]:
            next_groups.append(key)
        if len(calib_candidate_indices) >= candidate_target:
            break
    active_groups = next_groups

# --- Preprocesar y recolectar ---
calib_audios = []
calib_metadata = []
rejected = 0
current_offset = 0

for local_idx in tqdm(calib_candidate_indices, desc="Preprocesando calibracion"):
    if len(calib_audios) >= N_CALIB_SAMPLES:
        break
    sample = calib_source[local_idx]
    arr = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    processed = preprocess_audio(arr, sr)
    if processed is None:
        rejected += 1
        continue

    n_samples = len(processed)
    calib_audios.append(torch.from_numpy(processed))
    calib_metadata.append({
        "local_idx": local_idx,
        "sentence": sample["sentence"],
        "offset_start": current_offset,
        "offset_end": current_offset + n_samples,
        "n_samples": n_samples,
        "duration_s": n_samples / TARGET_SR,
        "gender": sample.get("gender", ""),
        "age": sample.get("age", ""),
        "accent": sample.get("accent", ""),
        "client_id": sample.get("client_id", ""),
    })
    current_offset += n_samples

print(f"\nMuestras procesadas : {len(calib_audios)}")
print(f"Muestras rechazadas : {rejected}")
print(f"Total de muestras de audio : {current_offset:,} ({current_offset/TARGET_SR:.1f}s)")
if len(calib_audios) != N_CALIB_SAMPLES:
    raise RuntimeError(f"Calibracion incompleta: {len(calib_audios)}/{N_CALIB_SAMPLES}")

[S1-06] Construccion de calibracion estratificada
Seleccionando 512 muestras de calibracion...


Preprocesando calibracion:  80%|████████  | 512/640 [00:03<00:00, 151.74it/s]


Muestras procesadas : 512
Muestras rechazadas : 0
Total de muestras de audio : 73,341,216 (3055.9s)


In [8]:
print("[S1-07] Guardado del tensor de calibracion")
# --- Concatenar en tensor unificado y guardar ---
calibration_tensor = torch.cat(calib_audios, dim=0)

tensor_path = DATA_DIR / "calibration_tensor.pt"
torch.save(calibration_tensor, tensor_path)

metadata_path = DATA_DIR / "calibration_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump({
        "dataset_id": DATASET_ID,
        "dataset_config": DATASET_CONFIG,
        "split_origin": "validation",
        "target_sr": TARGET_SR,
        "target_db_fs": TARGET_DB_FS,
        "n_samples": len(calib_audios),
        "total_audio_samples": calibration_tensor.shape[0],
        "total_duration_s": calibration_tensor.shape[0] / TARGET_SR,
        "records": calib_metadata,
    }, f, indent=2, ensure_ascii=False)

size_mb = tensor_path.stat().st_size / (1024 * 1024)
print(f"Tensor de calibracion : {tensor_path}")
print(f"Shape                 : {calibration_tensor.shape}")
print(f"Tamaño en disco       : {size_mb:.1f} MB")
print(f"Metadatos             : {metadata_path}")

[S1-07] Guardado del tensor de calibracion
Tensor de calibracion : /home/alfrog/projects/VibeVoice_Optimization/data/calibration_tensor.pt
Shape                 : torch.Size([73341216])
Tamaño en disco       : 279.8 MB
Metadatos             : /home/alfrog/projects/VibeVoice_Optimization/data/calibration_metadata.json


In [9]:
print("[S1-08] Verificacion del set de calibracion")
# --- Verificacion de carga ---
print("=== Verificacion del Set de Calibracion ===\n")

loaded_tensor = torch.load(DATA_DIR / "calibration_tensor.pt", weights_only=True)
with open(DATA_DIR / "calibration_metadata.json", "r") as f:
    loaded_meta = json.load(f)

print(f"Tensor cargado  : {loaded_tensor.shape}, dtype={loaded_tensor.dtype}")
print(f"Duracion total  : {loaded_meta['total_duration_s']:.1f}s")
print(f"Numero de clips : {loaded_meta['n_samples']}")
print(f"\nPrimeros 3 clips:")
for rec in loaded_meta["records"][:3]:
    print(f"  idx={rec['local_idx']:>6} | {rec['duration_s']:.2f}s | "
          f"\"{rec['sentence'][:60]}...\"")

# --- Verificar offsets ---
for i, rec in enumerate(loaded_meta["records"]):
    start, end = rec["offset_start"], rec["offset_end"]
    segment = loaded_tensor[start:end]
    assert len(segment) == rec["n_samples"], f"Offset mismatch en indice {i}"

print("\nVerificacion de offsets: OK")

[S1-08] Verificacion del set de calibracion
=== Verificacion del Set de Calibracion ===

Tensor cargado  : torch.Size([73341216]), dtype=torch.float32
Duracion total  : 3055.9s
Numero de clips : 512

Primeros 3 clips:
  idx= 15661 | 9.94s | "Para esta función poseen fuertes músculos especiales en el c..."
  idx= 15834 | 6.84s | "Sonny Parsons, conocido artísticamente como Sonny Parsons...."
  idx=   393 | 6.10s | "Nada que interese a un hombre del espacio. Sólo a los juguet..."

Verificacion de offsets: OK


---
# Sprint 1.5: Fine-Tuning Monolingue Espanol con LoRA

> **Prerrequisitos:** Requiere haber ejecutado **Sprint 1 completo** (celdas Bloque 0 hasta TSK-1.4).
> Variables necesarias: `cv17_all`, `train_dataset`, `val_dataset`, `DATA_DIR`, `WEIGHTS_DIR`, `PROJECT_ROOT`.

**Objetivo corregido:** reparar la precision linguistica del checkpoint VibeVoice-ES
existente mediante un segundo LoRA CE-only. El resultado se escribe como
**VibeVoice-ES-Corrected** sin sobrescribir la linea base anterior.

**Estrategia:**
- LoRA rank=8 sobre proyecciones Q/K/V/O + gate/up/down del LLM (Qwen2.5)
- Diffusion head, connectors y tokenizadores congelados
- `voice_prompt_drop_rate=1.0`: no se generan prompts desde el propio target
- `ce_loss_weight=1.0` y `diffusion_loss_weight=0.0` para correccion linguistica
- Desarrollo speaker-disjoint; validation nativo sigue reservado para PTQ
- `per_device_train_batch_size=1`, `gradient_accumulation_steps=64` para 8GB VRAM

> **Nota:** El entrenamiento ejecuta en terminal via `python -m vibevoice.finetune.train_vibevoice`.
> Las celdas de este bloque preparan los datos y generan el script de lanzamiento.
> La ejecucion del entrenamiento (~7-15h) se realiza fuera del notebook.

---
## TSK-1.5.1: Formatear Dataset al Esquema VibeVoice

El script de fine-tuning espera:
- Columna `text` con formato `"Speaker 1: {sentence}"`
- Columna `audio` como path a archivo de audio (str) o dict con array+sampling_rate
- Sin `voice_prompts`: la correccion es linguistica y usa drop rate 1.0

Guardamos los datos formateados como JSONL para pasarlos con `--train_jsonl` / `--validation_jsonl`.
Se crea un desarrollo speaker-disjoint a partir de `train` y esos hablantes se
excluyen del entrenamiento. El split nativo `validation` queda reservado para PTQ.
El texto se normaliza sin eliminar tildes, dieresis, `ñ`, `¿` ni `¡`.

In [10]:
print("[S1.5-01] Creacion de train/dev speaker-disjoint")
import json
import re
import unicodedata
from collections import defaultdict
from pathlib import Path

FINETUNE_DATA_DIR = DATA_DIR / "finetune"
FINETUNE_DATA_DIR.mkdir(parents=True, exist_ok=True)

def normalize_spanish_text(text: str) -> str:
    text = unicodedata.normalize("NFC", str(text or ""))
    text = "".join(ch for ch in text if ch.isprintable())
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"([.!?])\1+", r"\1", text)
    return text

# Desarrollo speaker-disjoint: ningun client_id de dev permanece en train.
N_VAL_TARGET = 2000
SPLIT_SEED = 42
speaker_indices = defaultdict(list)
for idx, client_id in enumerate(train_dataset["client_id"]):
    if client_id:
        speaker_indices[client_id].append(idx)
speakers = list(speaker_indices)
random.Random(SPLIT_SEED).shuffle(speakers)
val_indices_set = set()
val_speakers = []
for speaker in speakers:
    val_speakers.append(speaker)
    val_indices_set.update(speaker_indices[speaker])
    if len(val_indices_set) >= N_VAL_TARGET:
        break
assert val_indices_set and (set(range(len(train_dataset))) - val_indices_set).isdisjoint(val_indices_set)

def make_finetune_record(sample):
    text = normalize_spanish_text(sample.get("sentence", ""))
    audio_path = sample["audio"].get("path")
    if len(text) < 2 or not audio_path:
        return None
    return {
        "text": f"Speaker 1: {text}",
        "original_text": sample.get("sentence", ""),
        "audio": audio_path,
        "client_id": sample.get("client_id", ""),
    }

print("Formateando train/dev speaker-disjoint a JSONL...")
train_jsonl_path = FINETUNE_DATA_DIR / "cv17_es_train.jsonl"

val_jsonl_path = FINETUNE_DATA_DIR / "cv17_es_val.jsonl"
train_written = val_written = rejected = 0
with open(train_jsonl_path, "w", encoding="utf-8") as train_file, \
     open(val_jsonl_path, "w", encoding="utf-8") as val_file:
    for i in tqdm(range(len(train_dataset)), desc="Escribiendo train JSONL"):
        sample = train_dataset[i]
        record = make_finetune_record(sample)
        if record is None:
            rejected += 1
            continue
        if i in val_indices_set:
            val_file.write(json.dumps(record, ensure_ascii=False) + "\n")
            val_written += 1
        else:
            train_file.write(json.dumps(record, ensure_ascii=False) + "\n")
            train_written += 1

split_metadata = {
    "seed": SPLIT_SEED, "train_records": train_written,
    "validation_records": val_written, "validation_speakers": len(val_speakers),
    "rejected_records": rejected, "native_validation_reserved_for": "PTQ calibration",
}
with open(FINETUNE_DATA_DIR / "split_metadata.json", "w", encoding="utf-8") as file:
    json.dump(split_metadata, file, indent=2, ensure_ascii=False)
print(f"Train: {train_written:,} | Dev: {val_written:,} | Speakers dev: {len(val_speakers):,}")
print(f"Rechazados: {rejected:,} | Metadata: {FINETUNE_DATA_DIR / 'split_metadata.json'}")

[S1.5-01] Creacion de train/dev speaker-disjoint
Formateando train/dev speaker-disjoint a JSONL...


Escribiendo train JSONL: 100%|██████████| 336846/336846 [22:50<00:00, 245.80it/s]


Train: 334,237 | Dev: 2,609 | Speakers dev: 34
Rechazados: 0 | Metadata: /home/alfrog/projects/VibeVoice_Optimization/data/finetune/split_metadata.json


---
## TSK-1.5.2: Script de Fine-Tuning para 8GB VRAM

Parametros optimizados para RTX 4060 Ti (8GB):

| Parametro | Valor | Justificacion |
|-----------|-------|---------------|
| `per_device_train_batch_size` | 1 | Limitar VRAM de activaciones |
| `gradient_accumulation_steps` | 64 | Batch efectivo = 64 |
| `bf16` | True | Mixed precision (Ada Lovelace) |
| `ddpm_batch_mul` | 1 | Minimo; sin amplificacion de difusion |
| `lora_r` | 8 | Rank estandar para adaptacion |
| `learning_rate` | 1.0e-5 | Conservador para segundo LoRA |
| `num_train_epochs` | 1 | ~5,263 pasos (~7-15h) |
| `voice_prompt_drop_rate` | 1.0 | Evita prompt extraido del target |

El script generado se guarda en `scripts/run_finetune_es.sh`.

In [15]:
print("[S1.5-02] Validacion del script de entrenamiento")
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

VENV_PYTHON = "/home/alfrog/micromamba/envs/vibevoice/bin/python"
MODEL_PATH = str(WEIGHTS_DIR / "vibevoice-1.5b-es")
OUTPUT_DIR = str(PROJECT_ROOT / "outputs" / "finetune_vibevoice_es_correction")
TRAIN_JSONL = str(FINETUNE_DATA_DIR / "cv17_es_train.jsonl")
VAL_JSONL = str(FINETUNE_DATA_DIR / "cv17_es_val.jsonl")

train_script = f"""#!/bin/bash
# ============================================================
# Sprint 1.5: Correccion linguistica VibeVoice-ES (segundo LoRA)
# Hardware: RTX 4060 Ti 8GB VRAM
# Diffusion head y connectors congelados; objetivo CE-only
# ============================================================
set -e

cd {PROJECT_ROOT}

# Inyectar VibeVoice_repo en PYTHONPATH
export PYTHONPATH="{PROJECT_ROOT}/VibeVoice_repo:$PYTHONPATH"

echo "=== Iniciando Fine-Tuning VibeVoice-ES ==="
echo "Modelo base : {MODEL_PATH}"
echo "Train JSONL : {TRAIN_JSONL}"
echo "Val JSONL   : {VAL_JSONL}"
echo "Output      : {OUTPUT_DIR}"
echo ""

RESUME_ARGS=()
LATEST_CHECKPOINT=$(find "{OUTPUT_DIR}" -maxdepth 1 -type d -name 'checkpoint-*' -print 2>/dev/null | sort -V | tail -n 1)
if [[ -n "$LATEST_CHECKPOINT" ]]; then
    echo "Reanudando desde: $LATEST_CHECKPOINT"
    RESUME_ARGS=(--resume_from_checkpoint "$LATEST_CHECKPOINT")
fi

# Limpiar cache CUDA antes de empezar
{VENV_PYTHON} -c "import torch; torch.cuda.empty_cache(); free, total = torch.cuda.mem_get_info(); print(f'VRAM disponible: {{free/1024**3:.2f}}/{{total/1024**3:.2f}} GB')"

{VENV_PYTHON} -m vibevoice.finetune.train_vibevoice \\
    --model_name_or_path {MODEL_PATH} \\
    --train_jsonl {TRAIN_JSONL} \\
    --validation_jsonl {VAL_JSONL} \\
    --output_dir {OUTPUT_DIR} \\
    --text_column_name text \\
    --audio_column_name audio \\
    --per_device_train_batch_size 1 \\
    --gradient_accumulation_steps 64 \\
    --learning_rate 2.5e-5 \\
    --lr_scheduler_type cosine \\
    --warmup_ratio 0.03 \\
    --num_train_epochs 1 \\
    --logging_steps 50 \\
    --save_steps 500 \\
    --eval_steps 500 \\
    --save_total_limit 3 \\
    --bf16 True \\
    --do_train \\
    --do_eval \\
    --dataloader_num_workers 2 \\
    --remove_unused_columns False \\
    --gradient_checkpointing False \\
    --ddpm_batch_mul 1 \\
    --diffusion_loss_weight 1.0 \\
    --train_diffusion_head True \\
    --ce_loss_weight 1.0  # mas peso a precision textual \\
    --voice_prompt_drop_rate 1.0 \\
    --lora_r 8 \\
    --lora_alpha 32 \\
    --lora_dropout 0.05 \\
    --lora_target_modules q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj \\
    --max_grad_norm 0.8 \\
    --gradient_clipping \\
    --seed 42

echo ""
echo "=== Fine-Tuning completado ==="
echo "Checkpoint en: {OUTPUT_DIR}"
"""

script_path = SCRIPTS_DIR / "run_finetune_es.sh"
train_script = train_script.replace("--learning_rate 2.5e-5", "--learning_rate 1.0e-5")
train_script = train_script.replace("--logging_steps 50", "--logging_steps 50 \\n    --eval_strategy steps \\n    --save_strategy steps")
train_script = train_script.replace("--diffusion_loss_weight 1.0", "--diffusion_loss_weight 0.0")
train_script = train_script.replace("--train_diffusion_head True", "--train_diffusion_head False \\n    --train_connectors False")
train_script = train_script.replace("--ce_loss_weight 1.0  # mas peso a precision textual", "--ce_loss_weight 1.0")
train_script = train_script.replace("--voice_prompt_drop_rate 1.0", "--voice_prompt_drop_rate 1.0 \\n    --target_audio_dbfs -25.0 \\n    --augment_target_silence False")
train_script = train_script.replace("--seed 42", "--seed 42 \\n    \"${RESUME_ARGS[@]}\"")
# El script versionado es la fuente autoritativa; esta celda no lo sobrescribe
# con una plantilla historica.
if script_path.exists():
    train_script = script_path.read_text(encoding="utf-8")
script_path.write_text(train_script)
script_path.chmod(0o755)
import subprocess
subprocess.run(["bash", "-n", str(script_path)], check=True)

print(f"Script de entrenamiento generado: {script_path}")
print(f"\nPara ejecutar:\n  bash {script_path}")

[S1.5-02] Validacion del script de entrenamiento
Script de entrenamiento generado: /home/alfrog/projects/VibeVoice_Optimization/scripts/run_finetune_es.sh

Para ejecutar:
  bash /home/alfrog/projects/VibeVoice_Optimization/scripts/run_finetune_es.sh


In [ ]:
print("[S1.5-03] Ejecucion persistente del fine-tuning correctivo")
import json, sys
from pathlib import Path

notebook_cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in [notebook_cwd, notebook_cwd.parent] if (path / "scripts" / "run_finetune_es.sh").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"No se encontro la raiz del proyecto desde {notebook_cwd}")
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
WEIGHTS_DIR = PROJECT_ROOT / "weights"
train_script_path = SCRIPTS_DIR / "run_finetune_es.sh"
lora_path = OUTPUTS_DIR / "finetune_vibevoice_es_correction" / "lora"
adapter_config = lora_path / "adapter_config.json"
adapter_weights = [lora_path / "adapter_model.safetensors", lora_path / "adapter_model.bin"]
inputs = [
    train_script_path,
    PROJECT_ROOT / "data" / "finetune" / "cv17_es_train.jsonl",
    PROJECT_ROOT / "data" / "finetune" / "cv17_es_val.jsonl",
    WEIGHTS_DIR / "vibevoice-1.5b-es" / "config.json",
]
fingerprint = json.dumps([
    (path.name, path.stat().st_size, path.stat().st_mtime_ns) for path in inputs
])
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))
from persistent_notebook_job import run_persistent_job
run_persistent_job(
    PROJECT_ROOT,
    "s1-5-train",
    ["bash", str(train_script_path)],
    required_outputs=[adapter_config],
    fingerprint=fingerprint,
)
assert adapter_config.is_file(), f"Falta {adapter_config}"
assert any(path.is_file() for path in adapter_weights), f"Faltan pesos LoRA en {lora_path}"
print(f"[S1.5-03] LoRA correctivo validado: {lora_path}")


---
## TSK-1.5.3: Merge de LoRA al Modelo Base

Al finalizar, solo el segundo adaptador LoRA se fusiona sobre VibeVoice-ES.
El diffusion head congelado no se guarda ni se mergea. El resultado BF16 se
escribe en `weights/vibevoice-1.5b-es-corrected/`. La celda `S1.5-04` valida
el script y `S1.5-05` lo ejecuta y comprueba el checkpoint antes de continuar.

In [ ]:
print("[S1.5-04] Validacion del script de merge")
MERGE_OUTPUT = str(WEIGHTS_DIR / "vibevoice-1.5b-es-corrected")

merge_script = f"""#!/bin/bash
# ============================================================
# Sprint 1.5: Merge LoRA -> VibeVoice-ES
# ============================================================
set -e

cd {PROJECT_ROOT}

# Inyectar VibeVoice_repo en PYTHONPATH
export PYTHONPATH="{PROJECT_ROOT}/VibeVoice_repo:$PYTHONPATH"

echo "=== Mergeando LoRA + Diffusion Head al modelo base ==="
echo "Checkpoint LoRA : {OUTPUT_DIR}"
echo "Modelo base     : {MODEL_PATH}"
echo "Output (ES)     : {MERGE_OUTPUT}"
echo ""

{VENV_PYTHON} -m vibevoice.scripts.merge_vibevoice_models \\
    --base_model_path {MODEL_PATH} \\
    --checkpoint_path {OUTPUT_DIR}/lora \\
    --output_path {MERGE_OUTPUT} \\
    --output_format safetensors

echo ""
echo "=== VibeVoice-ES generado en: {MERGE_OUTPUT} ==="
ls -lh {MERGE_OUTPUT}/*.safetensors 2>/dev/null || echo "(verificar estructura)"
"""

merge_script_path = SCRIPTS_DIR / "merge_es_checkpoint.sh"
merge_script = merge_script.replace("--output_format safetensors", "--output_format safetensors \\n    --output_dtype bfloat16")
if merge_script_path.exists():
    merge_script = merge_script_path.read_text(encoding="utf-8")
merge_script_path.write_text(merge_script)
merge_script_path.chmod(0o755)
subprocess.run(["bash", "-n", str(merge_script_path)], check=True)

print(f"Script de merge generado: {merge_script_path}")
print(f"\nPara ejecutar (despues del entrenamiento):\n  bash {merge_script_path}")

In [ ]:
print("[S1.5-05] Ejecucion del merge LoRA correctivo")
from pathlib import Path
import sys

notebook_cwd = Path.cwd().resolve()
project_candidates = [notebook_cwd, notebook_cwd.parent]
PROJECT_ROOT = next(
    (path for path in project_candidates if (path / "scripts" / "merge_es_checkpoint.sh").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"No se encontro la raiz del proyecto desde {notebook_cwd}")
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
WEIGHTS_DIR = PROJECT_ROOT / "weights"

merge_script_path = SCRIPTS_DIR / "merge_es_checkpoint.sh"
corrected_path = WEIGHTS_DIR / "vibevoice-1.5b-es-corrected"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))
from persistent_notebook_job import run_persistent_job
run_persistent_job(
    PROJECT_ROOT,
    "s1-5-merge",
    ["bash", str(merge_script_path)],
    required_outputs=[corrected_path / "config.json"],
)

assert (corrected_path / "config.json").is_file(), f"Falta config.json en {corrected_path}"
weight_files = sorted(corrected_path.glob("*.safetensors"))
assert weight_files, f"No se generaron pesos safetensors en {corrected_path}"
print(f"[S1.5-05] Checkpoint corregido validado: {len(weight_files)} shard(s) en {corrected_path}")


---
## TSK-1.5.4: Validacion del Modelo Fine-Tuneado

Verificamos que **VibeVoice-ES-Corrected** carga en BF16 y registramos VRAM y
tamano. La cuantizacion no debe regenerarse hasta superar el gate WER/CER.

In [ ]:
print("[S1.5-06] Carga del checkpoint VibeVoice-ES-Corrected")
import gc, sys
from pathlib import Path
notebook_cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in [notebook_cwd, notebook_cwd.parent] if (path / "VibeVoice_repo").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"No se encontro la raiz del proyecto desde {notebook_cwd}")
WEIGHTS_DIR = PROJECT_ROOT / "weights"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
repo_path = PROJECT_ROOT / "VibeVoice_repo"
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))

import torch
import transformers
if hasattr(transformers, "CONFIG_MAPPING"):
    transformers.CONFIG_MAPPING._extra_content.pop("vibevoice_acoustic_tokenizer", None)
    if hasattr(transformers.CONFIG_MAPPING, "_mapping"):
        transformers.CONFIG_MAPPING._mapping.pop("vibevoice_acoustic_tokenizer", None)

from vibevoice.modular.modeling_vibevoice import VibeVoiceForConditionalGeneration

MODEL_ES_PATH = WEIGHTS_DIR / "vibevoice-1.5b-es-corrected"
required_files = [MODEL_ES_PATH / "config.json"]
if not all(path.is_file() for path in required_files) or not any(MODEL_ES_PATH.glob("*.safetensors")):
    lora_path = OUTPUTS_DIR / "finetune_vibevoice_es_correction" / "lora"
    if not lora_path.is_dir():
        raise FileNotFoundError(
            "Falta el LoRA correctivo. Ejecute S1.5-03 en una terminal y espere a que termine: "
            "bash scripts/run_finetune_es.sh"
        )
    raise FileNotFoundError(
        "Falta el checkpoint corregido. Ejecute S1.5-05 en una terminal: "
        "bash scripts/merge_es_checkpoint.sh"
    )

print("Cargando VibeVoice-ES-Corrected...")
if "model_es" in globals():
    del model_es
gc.collect()
torch.cuda.empty_cache()
model_es = VibeVoiceForConditionalGeneration.from_pretrained(
    MODEL_ES_PATH,
    torch_dtype=torch.bfloat16,
    device_map={"": "cuda:0"},
    attn_implementation="sdpa",
    local_files_only=True,
)

vram_es = torch.cuda.memory_allocated() / 1024**3
print(f"VibeVoice-ES-Corrected cargado. VRAM: {vram_es:.2f} GB")

# --- Verificacion de tamano en disco ---
import pathlib
model_size_mb = sum(f.stat().st_size for f in pathlib.Path(MODEL_ES_PATH).rglob("*.safetensors")) / (1024**2)
print(f"Tamano BF16 en disco: {model_size_mb:.1f} MB")

if vram_es < 6.0:
    print("VRAM OK - margen suficiente para cuantizacion.")
else:
    print("ADVERTENCIA: VRAM elevada. Verificar que el merge fue exitoso.")

In [ ]:
print("[S1.5-07] Gate linguistico espanol antes de cuantizar")
import gc, json, sys
from pathlib import Path

notebook_cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in [notebook_cwd, notebook_cwd.parent] if (path / "scripts" / "validate_vibevoice_es_quality.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"No se encontro la raiz del proyecto desde {notebook_cwd}")
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
WEIGHTS_DIR = PROJECT_ROOT / "weights"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
MODEL_ES_PATH = WEIGHTS_DIR / "vibevoice-1.5b-es-corrected"
quality_dir = OUTPUTS_DIR / "sprint1_5_quality_gate"
source_files = [MODEL_ES_PATH / "config.json", *sorted(MODEL_ES_PATH.glob("*.safetensors"))]
source_fingerprint = json.dumps([(p.name, p.stat().st_size, p.stat().st_mtime_ns) for p in source_files])
if "model_es" in globals():
    del model_es
gc.collect()
if "torch" in globals():
    torch.cuda.empty_cache()
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))
from persistent_notebook_job import run_persistent_job
run_persistent_job(
    PROJECT_ROOT,
    "s1-5-quality-gate",
    [sys.executable, str(SCRIPTS_DIR / "validate_vibevoice_es_quality.py"),
     "--model", str(MODEL_ES_PATH), "--output", str(quality_dir), "--max-wer", "0.30"],
    required_outputs=[quality_dir / "metrics.json"],
    fingerprint=source_fingerprint,
)
quality_metrics = json.loads((quality_dir / "metrics.json").read_text(encoding="utf-8"))
assert quality_metrics["passed"], quality_metrics
print(f"WER medio={quality_metrics['wer']:.4f} | CER={quality_metrics['cer']:.4f}")


## Construccion selectiva de Sprints 2-4

In [ ]:
print("La extraccion selectiva esta integrada en los builds S2-S4; no hay fase standalone manual.")


---
# Sprint 2: GGUF IQ4_NL selectivo con runtime C++

> Implementacion reproducible sobre CrispASR `v0.8.23`, no sobre `ggc v6`.
> Solo las 196 matrices de atencion/MLP de Qwen se almacenan como IQ4_NL.
> Tokenizadores, conectores, diffusion head y embeddings permanecen en F16/F32.
> Salida: `weights/vibevoice-1.5b-es-iq4_nl/vibevoice-1.5b-iq4_nl.gguf`.

El artefacto publico de gguf-org mide 3.536 GB y `ggc v6` vuelve a PyTorch; no respalda la afirmacion de menos de 1.2 GB. Este sprint usa el runtime GGML/C++ real de CrispASR y valida el tipo de cada tensor antes de aceptar el archivo.


In [ ]:
print("[S2-01] Preflight GGUF IQ4_NL selectivo")
import json
from pathlib import Path

notebook_cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [notebook_cwd, notebook_cwd.parent] if (p / "scripts").is_dir()), None)
if PROJECT_ROOT is None: raise FileNotFoundError(f"No se encontro la raiz desde {notebook_cwd}")
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
WEIGHTS_DIR = PROJECT_ROOT / "weights"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
MODEL_ES_PATH = WEIGHTS_DIR / "vibevoice-1.5b-es-corrected"
IQ4_OUTPUT = WEIGHTS_DIR / "vibevoice-1.5b-es-iq4_nl"
IQ4_MODEL = IQ4_OUTPUT / "vibevoice-1.5b-iq4_nl.gguf"
assert (MODEL_ES_PATH / "config.json").is_file(), (
    "Falta el checkpoint corregido. Complete S1.5-03, S1.5-05 y S1.5-07."
)
with (MODEL_ES_PATH / "config.json").open(encoding="utf-8") as f:
    cfg = json.load(f)
import sys
if str(SCRIPTS_DIR) not in sys.path: sys.path.insert(0, str(SCRIPTS_DIR))
from quality_gate import validate_quality_gate
quality_gate = validate_quality_gate(MODEL_ES_PATH, OUTPUTS_DIR / "sprint1_5_quality_gate" / "metrics.json")
decoder = cfg["decoder_config"]
assert decoder["hidden_size"] == 1536
assert decoder["num_hidden_layers"] == 28
assert decoder["num_attention_heads"] == 12
print("Arquitectura compatible: Qwen2 1.5B, 28 capas, 196 matrices objetivo")
print("Cota inferior del artefacto selectivo: ~3.28 GiB; <1.2 GB no es posible con audio F16")


In [ ]:
print("[S2-02] Conversion y cuantizacion IQ4_NL")
import sys
if str(SCRIPTS_DIR) not in sys.path: sys.path.insert(0, str(SCRIPTS_DIR))
from persistent_notebook_job import run_persistent_job
run_persistent_job(
    PROJECT_ROOT, "s2-iq4-nl-build",
    ["bash", str(SCRIPTS_DIR / "build_vibevoice_iq4_nl.sh"), str(MODEL_ES_PATH), str(IQ4_OUTPUT)],
    required_outputs=[IQ4_MODEL],
    fingerprint=json.dumps(quality_gate["source_hashes"], sort_keys=True),
)
assert IQ4_MODEL.is_file()
print(f"Modelo guardado: {IQ4_MODEL} ({IQ4_MODEL.stat().st_size / 1e9:.3f} GB)")


In [ ]:
print("[S2-03] Smoke test del runtime GGML/C++")
IQ4_SMOKE_WAV = OUTPUTS_DIR / "sprint2_iq4_nl" / "smoke-es.wav"
run_persistent_job(
    PROJECT_ROOT, "s2-iq4-nl-smoke",
    ["bash", str(SCRIPTS_DIR / "smoke_vibevoice_iq4_nl.sh"), str(IQ4_MODEL), str(IQ4_SMOKE_WAV)],
    required_outputs=[IQ4_SMOKE_WAV],
)
assert IQ4_SMOKE_WAV.is_file() and IQ4_SMOKE_WAV.stat().st_size > 44
print(f"Audio nativo guardado: {IQ4_SMOKE_WAV}")


---
# Sprint 3: GPTQ W4 g128 selectivo y recargable

> GPTQModel 2.2.0 + Transformers 4.51.3 en el entorno `vibevoice`.
> Solo las 196 proyecciones Qwen usan GPTQ W4; audio, connectors y diffusion head quedan BF16.
> El artefacto se acepta solo tras recargar los `QuantLinear` empaquetados en un proceso limpio.
> AutoGPTQ no se usa porque esta archivado y no mantiene Python 3.12/CUDA 13.


In [ ]:
print("[S3-01] Preflight GPTQ selectivo")
import json
import sys
from pathlib import Path

import gptqmodel
import transformers

notebook_cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [notebook_cwd, notebook_cwd.parent] if (p / "scripts").is_dir()), None)
if PROJECT_ROOT is None: raise FileNotFoundError(f"No se encontro la raiz desde {notebook_cwd}")
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
WEIGHTS_DIR = PROJECT_ROOT / "weights"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
DATA_DIR = PROJECT_ROOT / "data"
MODEL_ES_PATH = WEIGHTS_DIR / "vibevoice-1.5b-es-corrected"
GPTQ_OUTPUT = WEIGHTS_DIR / "vibevoice-1.5b-es-gptq"
GPTQ_METADATA = DATA_DIR / "calibration_metadata.json"
assert gptqmodel.__version__ == "2.2.0", (
    "Ejecute bash scripts/setup_gptqmodel.sh"
)
assert transformers.__version__ == "4.51.3"
assert (MODEL_ES_PATH / "config.json").is_file(), (
    "Falta el checkpoint corregido aprobado por S1.5-07"
)
assert GPTQ_METADATA.is_file()
with GPTQ_METADATA.open(encoding="utf-8") as file:
    calibration_info = json.load(file)
assert calibration_info["split_origin"] == "validation"
if str(SCRIPTS_DIR) not in sys.path: sys.path.insert(0, str(SCRIPTS_DIR))
from quality_gate import validate_quality_gate
quality_gate = validate_quality_gate(MODEL_ES_PATH, OUTPUTS_DIR / "sprint1_5_quality_gate" / "metrics.json")
print("GPTQModel 2.2.0 + Triton; objetivo: 196 proyecciones Qwen")


In [ ]:
print("[S3-02] Extraccion BF16, calibracion y cuantizacion GPTQ")
import sys
if str(SCRIPTS_DIR) not in sys.path: sys.path.insert(0, str(SCRIPTS_DIR))
from persistent_notebook_job import run_persistent_job
run_persistent_job(
    PROJECT_ROOT, "s3-gptq-build", [
        "bash", str(SCRIPTS_DIR / "build_vibevoice_gptq.sh"),
        str(MODEL_ES_PATH), str(GPTQ_OUTPUT), str(GPTQ_METADATA),
    ],
    required_outputs=[GPTQ_OUTPUT / "manifest.json"],
    fingerprint=json.dumps(quality_gate["source_hashes"], sort_keys=True),
)
with (GPTQ_OUTPUT / "manifest.json").open(encoding="utf-8") as file:
    gptq_manifest = json.load(file)
assert gptq_manifest["status"] in {"structure_validated", "validated"}
assert gptq_manifest["validation"]["gptq_modules"] == 196
print(f"Modelo GPTQ guardado: {GPTQ_OUTPUT}")


In [ ]:
print("[S3-03] Recarga limpia y smoke TTS del artefacto GPTQ")
GPTQ_SMOKE_WAV = OUTPUTS_DIR / "sprint3_gptq" / "smoke-es.wav"
run_persistent_job(
    PROJECT_ROOT, "s3-gptq-smoke", [
        sys.executable, str(SCRIPTS_DIR / "smoke_vibevoice_gptq.py"),
        "--model", str(GPTQ_OUTPUT), "--output", str(GPTQ_SMOKE_WAV),
    ],
    required_outputs=[GPTQ_SMOKE_WAV, GPTQ_SMOKE_WAV.with_suffix(".json")],
    fingerprint=json.dumps(gptq_manifest["artifacts"], sort_keys=True),
)
assert GPTQ_SMOKE_WAV.is_file() and GPTQ_SMOKE_WAV.stat().st_size > 44
with GPTQ_SMOKE_WAV.with_suffix(".json").open(encoding="utf-8") as file:
    gptq_smoke = json.load(file)
assert gptq_smoke["gptq_modules"] == 196
with (GPTQ_OUTPUT / "manifest.json").open(encoding="utf-8") as file:
    assert json.load(file)["status"] == "validated"
print(json.dumps(gptq_smoke, indent=2))


---
# Sprint 4: AWQ W4A16 selectivo y recargable

> AutoAWQ 0.2.9: W4A16 asimetrico, zero_point=True, grupo 128, GEMM/Triton.
> Solo las 196 proyecciones Qwen se cuantizan; audio, connectors y diffusion quedan FP16.
> El artefacto se acepta tras validar pesos, metadatos, ejecucion de las 196 capas y WER.
> No se etiqueta como Marlin: el artefacto ncoder-ai real tambien es GEMM y este host no tiene awq_ext.


In [ ]:
print("[S4-01] Preflight AWQ selectivo")
import importlib.util
import json
import sys
from pathlib import Path
import awq
import torch
import transformers

notebook_cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [notebook_cwd, notebook_cwd.parent] if (p / "scripts").is_dir()), None)
if PROJECT_ROOT is None: raise FileNotFoundError(f"No se encontro la raiz desde {notebook_cwd}")
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
WEIGHTS_DIR = PROJECT_ROOT / "weights"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
DATA_DIR = PROJECT_ROOT / "data"
MODEL_ES_PATH = WEIGHTS_DIR / "vibevoice-1.5b-es-corrected"
AWQ_OUTPUT = WEIGHTS_DIR / "vibevoice-1.5b-es-awq"
AWQ_METADATA = DATA_DIR / "calibration_metadata.json"
assert awq.__version__ == "0.2.9", "Ejecute bash scripts/setup_autoawq.sh"
assert transformers.__version__ == "4.51.3"
assert torch.cuda.get_device_capability(0) == (8, 9)
assert importlib.util.find_spec("awq_ext") is None, "Sprint 4 usa Triton reproducible"
assert (MODEL_ES_PATH / "config.json").is_file(), (
    "Falta el checkpoint corregido aprobado por S1.5-07"
)
with AWQ_METADATA.open(encoding="utf-8") as file:
    calibration_info = json.load(file)
assert calibration_info["split_origin"] == "validation"
if str(SCRIPTS_DIR) not in sys.path: sys.path.insert(0, str(SCRIPTS_DIR))
from quality_gate import validate_quality_gate
quality_gate = validate_quality_gate(MODEL_ES_PATH, OUTPUTS_DIR / "sprint1_5_quality_gate" / "metrics.json")
print("AutoAWQ GEMM/Triton; objetivo: 196 proyecciones Qwen, audio FP16")


In [ ]:
print("[S4-02] Extraccion FP16, calibracion y cuantizacion AWQ")
import sys
if str(SCRIPTS_DIR) not in sys.path: sys.path.insert(0, str(SCRIPTS_DIR))
from persistent_notebook_job import run_persistent_job
run_persistent_job(
    PROJECT_ROOT, "s4-awq-build", [
        "bash", str(SCRIPTS_DIR / "build_vibevoice_awq.sh"),
        str(MODEL_ES_PATH), str(AWQ_OUTPUT), str(AWQ_METADATA),
    ],
    required_outputs=[AWQ_OUTPUT / "manifest.json"],
    fingerprint=json.dumps(quality_gate["source_hashes"], sort_keys=True),
)
with (AWQ_OUTPUT / "manifest.json").open(encoding="utf-8") as file:
    awq_manifest = json.load(file)
assert awq_manifest["status"] in {"structure_validated", "validated"}
assert awq_manifest["validation"]["awq_modules"] == 196
print(f"Modelo AWQ guardado: {AWQ_OUTPUT}")


In [ ]:
print("[S4-03] Recarga limpia y smoke TTS del artefacto AWQ")
AWQ_SMOKE_WAV = OUTPUTS_DIR / "sprint4_awq" / "smoke-es.wav"
run_persistent_job(
    PROJECT_ROOT, "s4-awq-smoke", [
        sys.executable, str(SCRIPTS_DIR / "smoke_vibevoice_awq.py"),
        "--model", str(AWQ_OUTPUT), "--output", str(AWQ_SMOKE_WAV),
    ],
    required_outputs=[AWQ_SMOKE_WAV, AWQ_SMOKE_WAV.with_suffix(".json")],
    fingerprint=json.dumps(awq_manifest["artifacts"], sort_keys=True),
)
assert AWQ_SMOKE_WAV.is_file() and AWQ_SMOKE_WAV.stat().st_size > 44
with AWQ_SMOKE_WAV.with_suffix(".json").open(encoding="utf-8") as file:
    awq_smoke = json.load(file)
assert awq_smoke["awq_modules"] == 196
with (AWQ_OUTPUT / "manifest.json").open(encoding="utf-8") as file:
    assert json.load(file)["status"] == "validated"
print(json.dumps(awq_smoke, indent=2))


---
# Sprint 5: INT8 Hibrido Selectivo via bitsandbytes

> Replica el enfoque de Fabio Sarracino sobre VibeVoice 1.5B: cuantiza unicamente
> las capas `nn.Linear` del decoder Qwen2 con `BitsAndBytesConfig(load_in_8bit=True)`.
> El cabezal de difusion (`prediction_head` en este fork), los tokenizers, los
> connectors y `lm_head` permanecen en FP16 para proteger los latentes acusticos.

No se usa split layout: convertir el `state_dict` INT8 a FP16 destruiria la
cuantizacion. `save_pretrained()` conserva directamente los pesos y metadatos bnb.


In [ ]:
print("[S5-INT8] Ejecucion persistente fuera del kernel")
import json, sys
from pathlib import Path

notebook_cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [notebook_cwd, notebook_cwd.parent] if (p / "scripts" / "sprint5_int8.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"No se encontro la raiz desde {notebook_cwd}")
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
MODEL_ES_PATH = PROJECT_ROOT / "weights" / "vibevoice-1.5b-es-corrected"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))
from quality_gate import validate_quality_gate
from persistent_notebook_job import run_persistent_job
quality_gate = validate_quality_gate(MODEL_ES_PATH, OUTPUTS_DIR / "sprint1_5_quality_gate" / "metrics.json")
metrics_path = OUTPUTS_DIR / "sprint5_int8" / "metrics.json"
run_persistent_job(
    PROJECT_ROOT,
    "s5-int8",
    [sys.executable, str(SCRIPTS_DIR / "sprint5_int8.py")],
    required_outputs=[metrics_path],
    workdir=PROJECT_ROOT / "notebooks",
    fingerprint=json.dumps(quality_gate["source_hashes"], sort_keys=True),
)
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
print(json.dumps(metrics, indent=2))


---
# Sprint 6: NF4 Hibrido + Double Quant via bitsandbytes

> Cuantiza unicamente las 196 proyecciones lineales del Qwen2 con NF4 y doble
> cuantizacion. `embed_tokens`, `lm_head` y todo el pipeline acustico permanecen
> en BF16; las multiplicaciones NF4 tambien usan BF16 como compute dtype.

`embed_tokens` es `nn.Embedding`, por lo que bitsandbytes no lo reemplazaria de
todos modos; se incluye en la lista para documentar y verificar la proteccion.
No se usa split layout ni se exporta un state dict dequantizado a FP16.


In [ ]:
print("[S6-NF4] Ejecucion persistente fuera del kernel")
import json, sys
from pathlib import Path

notebook_cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [notebook_cwd, notebook_cwd.parent] if (p / "scripts" / "sprint6_nf4.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"No se encontro la raiz desde {notebook_cwd}")
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
MODEL_ES_PATH = PROJECT_ROOT / "weights" / "vibevoice-1.5b-es-corrected"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))
from quality_gate import validate_quality_gate
from persistent_notebook_job import run_persistent_job
quality_gate = validate_quality_gate(MODEL_ES_PATH, OUTPUTS_DIR / "sprint1_5_quality_gate" / "metrics.json")
metrics_path = OUTPUTS_DIR / "sprint6_nf4" / "metrics.json"
run_persistent_job(
    PROJECT_ROOT,
    "s6-nf4",
    [sys.executable, str(SCRIPTS_DIR / "sprint6_nf4.py")],
    required_outputs=[metrics_path],
    workdir=PROJECT_ROOT / "notebooks",
    fingerprint=json.dumps(quality_gate["source_hashes"], sort_keys=True),
)
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
print(json.dumps(metrics, indent=2))


---
# Sprint 7: FP8 E4M3FN con Escalamiento Dinamico

> Las 196 proyecciones lineales de Qwen almacenan pesos FP8 escalados por capa.
> Cada forward escala dinamicamente las activaciones, ejecuta `torch._scaled_mm`
> FP8 nativo y devuelve BF16. Audio, embeddings y `lm_head` permanecen en BF16.

PyTorch reporta un maximo finito de 448 para `float8_e4m3fn`; se usa 240 como
margen conservador, no como limite del dtype. El loader custom esta en
`scripts/fp8_vibevoice.py`. El offload por modulos es opcional; este sprint no
implementa descarga circular del KV cache.


In [ ]:
print("[S7-FP8] Ejecucion persistente fuera del kernel")
import json, sys
from pathlib import Path

notebook_cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [notebook_cwd, notebook_cwd.parent] if (p / "scripts" / "sprint7_fp8.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"No se encontro la raiz desde {notebook_cwd}")
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
MODEL_ES_PATH = PROJECT_ROOT / "weights" / "vibevoice-1.5b-es-corrected"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))
from quality_gate import validate_quality_gate
from persistent_notebook_job import run_persistent_job
quality_gate = validate_quality_gate(MODEL_ES_PATH, OUTPUTS_DIR / "sprint1_5_quality_gate" / "metrics.json")
metrics_path = OUTPUTS_DIR / "sprint7_fp8" / "metrics.json"
run_persistent_job(
    PROJECT_ROOT,
    "s7-fp8",
    [sys.executable, str(SCRIPTS_DIR / "sprint7_fp8.py")],
    required_outputs=[metrics_path],
    workdir=PROJECT_ROOT / ".",
    fingerprint=json.dumps(quality_gate["source_hashes"], sort_keys=True),
)
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
print(json.dumps(metrics, indent=2))


In [ ]:
print("[S7-FP8] Benchmark integrado en el job persistente anterior.")


In [ ]:
print("[S7-FP8] ASR y metricas integrados en el job persistente anterior.")


---
# Sprint 8: Benchmark Final (7 Modelos + Estadistica)

> 200 muestras del split test de CV17.
> Metricas: VRAM, RTF, WER, CER, PESQ, MCD, Disco.
> Shapiro-Wilk + Wilcoxon + Graficos + Tabla LaTeX.


In [ ]:
# === Sprint 8: Benchmark Final (placeholder - se completa al tener todos los modelos) ===
print("Sprint 8 se ejecuta despues de generar los 7 modelos.")
print("Modelos: FP16, IQ4_NL, GPTQ, AWQ, INT8, NF4, FP8")
print("200 muestras del split test de CV17")
print("Metricas: VRAM, RTF, WER, CER, PESQ, MCD, Disco")
print("Stats: Shapiro-Wilk + Wilcoxon signed-rank")
print("Output: outputs/sprint8/ (CSV, graficos PNG, tabla LaTeX)")
